In [ ]:
import torch
import torchvision

# Create a vector of zeros of size 5
size = (128, 128)
#transform will resize the image to 128x128 and convert it to a tensor [so uniforms the data]
transform = torchvision.transforms.Compose([torchvision.transforms.Resize(size), torchvision.transforms.ToTensor()])
train_dataset = torchvision.datasets.Flowers102("/tmp/flowers", "train", transform=transform, download=True)
test_dataset = torchvision.datasets.Flowers102("/tmp/flowers", "test", transform=transform, download=True)

In [ ]:


train_dataset.__len__() # len(train_dataset[0]) is the same 
train_dataset.__getitem__(0) # train_dataset[0] is the same

#train_dataset is a list of transformed images and their labels. So train_dataset[0] is a tuple of (image, label)

(tensor([[[0.0471, 0.0706, 0.0745,  ..., 0.1255, 0.4667, 0.5647],
          [0.0667, 0.0667, 0.0549,  ..., 0.1333, 0.4824, 0.5647],
          [0.0824, 0.0745, 0.0549,  ..., 0.1451, 0.5059, 0.5686],
          ...,
          [0.1059, 0.1059, 0.0863,  ..., 0.5020, 0.4902, 0.4706],
          [0.1137, 0.1137, 0.1294,  ..., 0.5059, 0.4784, 0.4706],
          [0.1020, 0.1176, 0.1176,  ..., 0.5020, 0.4745, 0.4667]],
 
         [[0.0863, 0.1255, 0.1373,  ..., 0.1294, 0.3412, 0.3961],
          [0.0941, 0.1098, 0.1059,  ..., 0.1294, 0.3490, 0.3922],
          [0.0941, 0.0941, 0.0824,  ..., 0.1294, 0.3608, 0.3843],
          ...,
          [0.2000, 0.1804, 0.1333,  ..., 0.4235, 0.4118, 0.3922],
          [0.2118, 0.2039, 0.2000,  ..., 0.4275, 0.4039, 0.3922],
          [0.2078, 0.2196, 0.2196,  ..., 0.4196, 0.4078, 0.3765]],
 
         [[0.0314, 0.0392, 0.0353,  ..., 0.0863, 0.4745, 0.5961],
          [0.0392, 0.0353, 0.0235,  ..., 0.0980, 0.4902, 0.5922],
          [0.0431, 0.0353, 0.0235,  ...,

In [ ]:
#torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
#expects dataset
#expects batch size
#expects shuffle to be a boolean - usually best to do true
#can also do num_workers to speed up data loading by using multiple processes by making it more than zero
#can also do drop_last to drop the last batch if it's not full [11 images and batch size is 6 then drop the last 5 images] - usually best to do false
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

# to go through the data loader we can do a for loop
#batch is a tuple of (images, labels) where images is a tensor of shape (batch_size, channels, height, width) and labels is a tensor of shape (batch_size)
#for batch in train_loader:
    #print(batch)
    #break

#instead look at shape
for imgs, labels in train_loader:
    print(imgs.shape, labels.shape)
    break
#imgs.shape gives batch size [64 images], channels [3 colors], SizeA [128 - from initial variable of size], SizeB [128 - from initial variable of size]
#labels.Size gives batch size [64 since each image has a label]


torch.Size([64, 3, 128, 128]) torch.Size([64])


In [ ]:
class MyModel(torch.nn.Module):
    def __init__(self, layer_size=[512, 512, 512]):
        super().__init__()
        layers = []
        layers.append(torch.nn.Flatten())
        c = 128 * 128 * 3
        for s in layer_size:
            layers.append(torch.nn.Linear(c, s))
            layers.append(torch.nn.ReLU())
            c = s
        layers.append(torch.nn.Linear(c, 102))
        self.model = torch.nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

model = MyModel(layer_size=[])
model.cuda()

loss_fn = torch.nn.CrossEntropyLoss()

optim = torch.optim.SGD(model.parameters(), lr=0.0001, momentum=0.9)

for epoch in range(10):
    for imgs, labels in train_loader:
        imgs, labels = imgs.cuda(), labels.cuda()
        pred = model(imgs)

        loss_val = loss_fn(pred, labels)

        optim.zero_grad()
        loss_val.backward()
        optim.step()
        print(loss_val.item())